# 08 Daily actionable insights

The daily endpoint is fetched once. Two tables use the same response to describe groups and unusual stocks.

Research data only, not investment advice. Each panel refreshes on a schedule; each panel shows its source date.

Free access grants no redistribution rights. **Do not republish the data you fetch.**

These are member research routes (beta). After stock-api release 4 is live, they are listed in /api/v1/catalog member_research_endpoints (from stock-api release 4).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'notebooks' / '_shared.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'notebooks'))
sys.path.insert(0, str(ROOT / 'python' / 'src'))
from _shared import fetch_once, fetch_panel, get_api_key, make_client, nested
client, catalogue = make_client()
api_key = get_api_key()
get_insights = fetch_once(lambda: client.member_daily_insights(api_key))

In [ ]:
group_columns = {
    'id': 'The numeric identifier for the statistical group.',
    'name': 'The display name for the group.',
    'plain_name': 'The plain-language description of the group.',
    'count': 'The number of stocks assigned to the group.',
    'share': 'The group share of all assigned stocks.',
    'sample_tickers': 'Example tickers from the group.',
    'churn': 'The share of group membership that changed from the prior run.',
}
groups_frame = fetch_panel('Daily stock groups', get_insights, lambda p: nested(p, 'data', 'map', 'groups') or [], group_columns, source_date=lambda p: nested(p, 'data', 'as_of_date'), status=lambda p: nested(p, 'data', 'status'), message=lambda p: nested(p, 'data', 'reason_code') or (None if nested(p, 'data', 'map', 'groups') else 'No daily groups returned.'))

In [ ]:
card_columns = {
    'card_id': 'The identifier for the narrative card.',
    'title': 'The display title for the card.',
    'body': 'The plain-language summary for the card.',
    'ticker': 'The ticker attached to this item when present.',
    'group': 'The group name attached to this item when present.',
    'reason': 'The plain-language reason attached to this item.',
    'score': 'The statistical score attached to this item.',
    'from_group': 'The prior group when the item changed groups.',
    'to_group': 'The new group when the item changed groups.',
    'low_coverage': 'Whether missing inputs affect the item.',
}
def card_rows(payload):
    return [{'card_id': card.get('id'), 'title': card.get('title'), 'body': card.get('body'), **(item or {})} for card in nested(payload, 'data', 'cards') or [] for item in card.get('items') or [None]]
cards_frame = fetch_panel('Daily narrative cards', get_insights, card_rows, card_columns, source_date=lambda p: nested(p, 'data', 'as_of_date'), status=lambda p: nested(p, 'data', 'status'), message=lambda p: nested(p, 'data', 'reason_code') or (None if nested(p, 'data', 'cards') else 'No daily cards returned.'))